In [ ]:
import chromadb , json, ollama

client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_collection("lufthansa")     # reopen the existing store

print("Loaded collection with", collection.count(), "docs")

Loaded collection with 185 docs


In [ ]:
PATH = "lufthansa_labeled.json"
labeled = json.load(open(PATH, encoding="utf-8"))

for d in labeled:
    if d["category"] == "opportunity":
        r = ollama.chat(model="llama3.1:8b", messages=[{"role": "user", "content":
            "Rate the business impact of this opportunity for Lufthansa. "
            "Answer with exactly ONE word — High, Medium, or Low.\n\n" + d["text"]}])
        ans = r["message"]["content"].strip().split()[0].strip(".,").capitalize()
        d["impact"] = ans if ans in ("High", "Medium", "Low") else "Medium"
        print(d["text"][:55], "->", d["impact"])

json.dump(labeled, open(PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("Saved impact for", sum(d["category"] == "opportunity" for d in labeled), "opportunities")

Where Lufthansa’s quadjets are flying this summer. As L -> Medium
Success Booking Lufthansa via Lifemiles, here are few t -> Medium
Barclays Lufthansa Miles & More Increased Sign Up Bonus -> Medium
Financial results | AIR FRANCE KLM. 1st Quarter results -> Medium
KLM Royal Dutch Airlines - Book flights online - KLM US -> Medium
Air France KLM targets high tech cargo with. - Air Carg -> Medium
Why EasyJet's Share Price Has Surged 21% in a Month — A -> Medium
Emirates boosts long-haul fleet with mega order - PaxEx -> Medium
Turkish Airlines - airBaltic Expansion. Turkish Airline -> Medium
Award-winning Lufthansa Allegris cabin now bookable for -> Medium
Media Library - newsroom.lufthansagroup.com. Lufthansa  -> Medium
Investor Relations - Lufthansa Group Investor Relations -> Medium
Saved impact for 12 opportunities


In [19]:
def ceo_agent(question, k=5):
    # 1. RETRIEVE evidence
    results   = collection.query(query_texts=[question], n_results=k)
    retrieved = results["documents"][0]
    metas     = results["metadatas"][0]
    context   = "\n\n".join(f"[{m['source']}] {doc}" for doc, m in zip(retrieved, metas))

    # 2. PROMPT
    system_prompt = """You are a strategic advisor to the CEO of Lufthansa.
Use ONLY the evidence provided — do not invent facts.
Return a JSON object with EXACTLY these keys:
- "recommendation": one clear strategic action (string)
- "justification": 1-2 sentences explaining WHY this recommendation follows from the evidence
- "supporting_evidence": list of 2-3 short evidence points from the context
- "expected_impact": expected business impact (string)
- "risk_level": one of "High", "Medium", "Low"
- "priority": one of "High", "Medium", "Low"
"""
    user_prompt = f"Evidence:\n{context}\n\nQuestion: {question}"

    # 3. GENERATE (structured JSON)
    response = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        format="json"
    )

    # 4. PARSE + attach question and evidence URLs
    rec = json.loads(response["message"]["content"])
    rec["question"] = question
    rec["sources"]  = [m["url"] for m in metas]
    return rec

In [20]:
questions = [
    "What are the major opportunities for Lufthansa?",
    "What are the biggest risks for Lufthansa?",
    "What are competitors doing?",
    "Which technologies or trends should Lufthansa management monitor?",
    "What strategic actions should Lufthansa prioritize?",
]

recommendations = []
for q in questions:
    print("Generating:", q)
    recommendations.append(ceo_agent(q))

json.dump(recommendations,
          open("recommendations.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("\n Saved", len(recommendations), "recommendations")

Generating: What are the major opportunities for Lufthansa?
Generating: What are the biggest risks for Lufthansa?
Generating: What are competitors doing?
Generating: Which technologies or trends should Lufthansa management monitor?
Generating: What strategic actions should Lufthansa prioritize?

 Saved 5 recommendations


CEO Briefing (Section 7)

In [21]:
def ceo_briefing(recommendations):
    # summarize the recommendations as input
    rec_summary = "\n".join(
        f"- {r['recommendation']} (priority {r['priority']}, risk {r['risk_level']})"
        for r in recommendations
    )

    system_prompt = """You are chief of staff to the CEO of Lufthansa.
Write a concise executive briefing as a JSON object with EXACTLY these 3 keys:
- "what_happened": a single plain-text string (2-3 sentences)
- "why_it_matters": a single plain-text string (2-3 sentences)
- "what_to_do_next": a single plain-text string (2-3 sentences)
Each value MUST be a plain string — NOT a nested object, dict, or list.
Base it ONLY on the recommendations provided. Do not invent facts."""

    user_prompt = f"Strategic recommendations:\n{rec_summary}\n\nWrite the CEO briefing."

    response = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        format="json"
    )
    return json.loads(response["message"]["content"])

In [ ]:
briefing = ceo_briefing(recommendations)
json.dump(briefing, open("ceo_briefing.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print(json.dumps(briefing, indent=2, ensure_ascii=False))

{
  "what_happened": "Recent customer surveys have revealed significant dissatisfaction with our service, impacting loyalty and revenue.",
  "why_it_matters": "These issues are eroding our market share and damaging our reputation, threatening long-term sustainability.",
  "what_to_do_next": "We will immediately implement the recommended strategies to improve customer experience, address satisfaction concerns, monitor the competitive landscape, integrate Lufthansa Cargo with global logistics networks, and invest in data collection from fuel-efficient aircraft."
}


### REWORK

In [23]:
from retrieval import semantic_search, bm25_search, hybrid_search

In [ ]:
def retrieve_evidence(query, k_each=5, final_k=5):
    """Run all 3 retrievers, pool results, dedup by URL, keep the best by consensus."""
    methods = {
        "semantic": semantic_search(query, k_each),
        "bm25":     bm25_search(query, k_each),
        "hybrid":   hybrid_search(query, k_each),
    }

    seen = {}                                   # url -> {doc, hits, rank_sum}
    for docs_list in methods.values():
        for rank, d in enumerate(docs_list):    # rank 0 = top of that method
            key = d["url"]
            if key not in seen:
                seen[key] = {"doc": d, "hits": 0, "rank_sum": 0}
            seen[key]["hits"]     += 1           # how many methods found it
            seen[key]["rank_sum"] += rank        # how high they ranked it

    # best = found by MOST methods, tie-break by best average rank
    ranked = sorted(seen.values(), key=lambda x: (-x["hits"], x["rank_sum"]))
    return [x["doc"] for x in ranked[:final_k]]

In [25]:
import ollama, json

def make_plan(goal):
    """Break the CEO's abstract goal into specific, searchable sub-questions."""
    system = (
        "You are a research planner for a strategic intelligence agent about Lufthansa. "
        "Break the user's question into 2-4 SPECIFIC, keyword-rich sub-questions "
        "that will retrieve good evidence from a news database. "
        'Return JSON exactly like: {"steps": ["...", "...", "..."]}'
    )
    res = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": goal},
        ],
        format="json",
    )
    return json.loads(res["message"]["content"])["steps"]

In [26]:
goal = "What are the biggest risks for Lufthansa?"
plan = make_plan(goal)

print("🧭 PLAN for:", goal)          # ← showing the agent's thinking (not a black box!)
for i, s in enumerate(plan, 1):
    print(f"   {i}. {s}")

🧭 PLAN for: What are the biggest risks for Lufthansa?
   1. What are the top 5 financial challenges facing Lufthansa in the current market?
   2. What are the most significant safety concerns related to Lufthansa's recent flight operations or fleet maintenance?
   3. How is Lufthansa addressing the increasing competition from low-cost carriers in Europe and the impact on its market share?
   4. What are the regulatory risks faced by Lufthansa due to changes in EU aviation laws or sanctions related to other countries?


In [29]:
def gather_evidence(goal, final_per_step=4):
    """Plan the goal, retrieve for each sub-question, pool + dedup the evidence."""
    plan = make_plan(goal)                              # 1. break goal into sub-questions
    print(f"🧭 PLANNING:{goal}")
    for s in plan:
        print("   -", s)

    pooled = []
    for sub_q in plan:                                  # 2. retrieve for EACH sub-question
        docs = retrieve_evidence(sub_q, final_k=final_per_step)
        print(f"   🔎 {sub_q[:45]}... → {len(docs)} docs")
        pooled.extend(docs)                             # add them all to one big list

    unique = list({d["url"]: d for d in pooled}.values())  # 3. dedup by URL
    print(f"\n📊 Pooled {len(pooled)} → {len(unique)} unique evidence docs")
    return unique

In [30]:
evidence = gather_evidence("What are the biggest risks for Lufthansa?")
print("\nSample:", evidence[0]["text"][:80])

🧭 PLANNING:What are the biggest risks for Lufthansa?
   - Lufthansa financial performance: What recent news articles mention potential declines in profit or revenue?
   - What are the major operational challenges facing Lufthansa, such as flight delays, cancellations, or maintenance issues?
   - How does Lufthansa compare to its competitors in terms of market share and passenger traffic?
   - What regulatory changes or economic trends may impact Lufthansa's operations and profitability?
   🔎 Lufthansa financial performance: What recent ... → 4 docs
   🔎 What are the major operational challenges fac... → 4 docs
   🔎 How does Lufthansa compare to its competitors... → 4 docs
   🔎 What regulatory changes or economic trends ma... → 4 docs

📊 Pooled 16 → 15 unique evidence docs

Sample: Lufthansa Group Posts Record Revenue, Profit Surge. Mar 6, 2026 · COLOGNE — The 
